In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

# Configuración global
np.random.seed(1989)
pd.set_option("display.float_format", '{:,.2f}'.format)

print("Bibliotecas importadas correctamente.")

Bibliotecas importadas correctamente.


In [4]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Cargar el dataset
df = pd.read_csv('muestra.csv')

# --- INGENIERÍA DE VARIABLES MEJORADA ---

# Estado y Segmento
df['Es_Reacondicionado'] = df['Nombre'].str.contains(
    r'(?:reacondicionado|renovado|refurbished|used|usado|caja abierta|open box)', 
    case=False, regex=True
).astype(int)

df['Es_Gaming'] = df['Nombre'].str.contains(
    r'(?:gaming|gamer|rtx|gtx|loq|victus|legion|rog|tuf|predator|katana|nitro|pavilion gaming|alienware|cyborg|thin 15|gf63|stealth|strix)', 
    case=False, regex=True
).astype(int)

# Líneas Profesionales y Premium
df['Es_Workstation'] = df['Nombre'].str.contains(
    r'(?:workstation|zbook|precision|thinkpad p|studio|probook|latitude|elitebook)', 
    case=False, regex=True
).astype(int)

df['Es_Linea_Premium'] = df['Nombre'].str.contains(
    r'(?:zenbook|macbook pro|spectre|xps|gram|copilot\+|omnibook|swift go|razer)', 
    case=False, regex=True
).astype(int)

# Pantalla y Marca
df['Es_Touch'] = df['Nombre'].str.contains(r'(?:touch|táctil|tactil|touchscreen|2 en 1|2-in-1)', case=False, regex=True).astype(int)
df['Es_OLED'] = df['Nombre'].str.contains(r'oled', case=False, regex=True).astype(int)
df['Es_Apple'] = ((df['Marca'].str.lower() == 'apple') | (df['Nombre'].str.contains(r'macbook', case=False, regex=True))).astype(int)

# Jerarquía Fina de GPU (Tiers 0 a 6)
def obtener_tier_gpu(nombre):
    nombre_lower = str(nombre).lower()
    if re.search(r'rtx\s*(?:4090|5090|3090)', nombre_lower):
        return 6
    elif re.search(r'rtx\s*(?:4080|5080|3080|a3000|a4000|a5000)', nombre_lower):
        return 5
    elif re.search(r'rtx\s*(?:4070|5070|3070|a2000)', nombre_lower):
        return 4
    elif re.search(r'rtx\s*(?:4060|5060|3060|a1000|a500)', nombre_lower):
        return 3
    elif re.search(r'rtx\s*(?:4050|3050|2050|1650)|gtx|radeon rx', nombre_lower):
        return 2
    elif re.search(r'iris|radeon|uhd|graphics|intel hd', nombre_lower):
        return 1
    return 0

df['GPU_Tier_Detallado'] = df['Nombre'].apply(obtener_tier_gpu)

# Jerarquía Fina de CPU (Tiers 1 a 6)
def obtener_tier_cpu(row):
    nombre = str(row['Nombre']).lower()
    proc = str(row['Procesador']).lower()
    if re.search(r'ultra 9|i9|ryzen 9|m2 max|m3 max|m3 pro|m2 pro', nombre):
        return 6
    elif re.search(r'ultra 7|i7|ryzen 7|ryzen ai 7|m1 max|m1 pro|m3|m2', nombre) or proc in ['core i7', 'ryzen 7']:
        return 5
    elif re.search(r'ultra 5|i5|ryzen 5|ryzen ai 5|m1', nombre) or proc in ['core i5', 'ryzen 5']:
        return 4
    elif re.search(r'i3|ryzen 3', nombre) or proc in ['core i3', 'ryzen 3']:
        return 3
    elif re.search(r'celeron|pentium|n150|n200|n100|n305|athlon|silver|chromebook', nombre):
        return 1
    return 2

df['CPU_Tier_Detallado'] = df.apply(obtener_tier_cpu, axis=1)

# Variables de interacción
df['RAM_GB'] = df['RAM_GB'].fillna(8)
df['Almacenamiento_GB'] = df['Almacenamiento_GB'].fillna(512)
df['Log2_RAM'] = np.log2(np.maximum(df['RAM_GB'], 1))
df['Log2_Almacenamiento'] = np.log2(np.maximum(df['Almacenamiento_GB'], 1))

df['RAM_x_CPU'] = df['RAM_GB'] * df['CPU_Tier_Detallado']
df['RAM_x_GPU'] = df['RAM_GB'] * df['GPU_Tier_Detallado']
df['Gaming_x_GPU'] = df['Es_Gaming'] * df['GPU_Tier_Detallado']

# One-hot encoding de Marcas
df_encoded = pd.get_dummies(df, columns=['Marca'], prefix='Marca', drop_first=True)

# OPCIÓN RECOMENDADA: Acotar a equipos de mercado masivo (<= $65,000 MXN)
df_filtrado = df_encoded[df_encoded['Precio_MXN'] <= 65000].copy()

feature_cols = [
    'RAM_GB', 'Almacenamiento_GB', 'Log2_RAM', 'Log2_Almacenamiento', 
    'CPU_Tier_Detallado', 'GPU_Tier_Detallado',
    'RAM_x_CPU', 'RAM_x_GPU', 'Gaming_x_GPU',
    'Es_Reacondicionado', 'Es_Gaming', 'Es_Workstation', 'Es_Linea_Premium',
    'Es_Touch', 'Es_OLED', 'Es_Apple'
] + [c for c in df_filtrado.columns if c.startswith('Marca_')]

X = df_filtrado[feature_cols]
y = df_filtrado['Precio_MXN']

# División Train / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelo Entrenado
rf_opt = RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=2, random_state=42)
rf_opt.fit(X_train, y_train)

y_pred_train = rf_opt.predict(X_train)
y_pred_test = rf_opt.predict(X_test)

print("=======================================================")
print("MODELO CON INGENIERÍA AVANZADA Y CONTROL DE OUTLIERS:")
print("=======================================================")
print(f"R² Entrenamiento: {r2_score(y_train, y_pred_train):.2f}")
print(f"R² Prueba:        {r2_score(y_test, y_pred_test):.2f}")
print(f"MAE (Error Promedio): ${mean_absolute_error(y_test, y_pred_test):,.2f} MXN")
print(f"RMSE:                 ${np.sqrt(mean_squared_error(y_test, y_pred_test)):,.2f} MXN")
print("=======================================================")

MODELO CON INGENIERÍA AVANZADA Y CONTROL DE OUTLIERS:
R² Entrenamiento: 0.81
R² Prueba:        0.70
MAE (Error Promedio): $3,811.28 MXN
RMSE:                 $5,659.87 MXN


In [5]:
# 1. Predicciones con el modelo optimizado
y_pred_opt = rf_opt.predict(X_test)

# 2. Reconstrucción de la tabla de análisis
df_residuales = df_filtrado.loc[X_test.index, ['Nombre', 'Precio_MXN', 'Es_Reacondicionado', 'Es_Gaming']].copy()
df_residuales['Precio_Justo_Modelo'] = y_pred_opt
df_residuales['Diferencia_MXN'] = df_residuales['Precio_MXN'] - df_residuales['Precio_Justo_Modelo']

# 3. Filtrar Gangas (Diferencia Negativa = Cuesta menos de lo que vale su hardware)
gangas = df_residuales[df_residuales['Diferencia_MXN'] < -2000].sort_values(by='Diferencia_MXN', ascending=True)

# 4. Filtrar Sobreprecios (Diferencia Positiva = Cobro extra por marca o línea)
sobreprecios = df_residuales[df_residuales['Diferencia_MXN'] > 2000].sort_values(by='Diferencia_MXN', ascending=False)

print("=======================================================================")
print("🎯 DETECTOR FINANCIERO: TOP OPORTUNIDADES Y SOBREPRECIOS DE MERCADO")
print("=======================================================================\n")

cols = ['Nombre', 'Precio_MXN', 'Precio_Justo_Modelo', 'Diferencia_MXN']

print("⬇️ TOP 5 GANGAS REALES (Mayor descuento frente al valor del hardware) ⬇️")
print(gangas[cols].head().round(2).to_string(index=False))

print("\n⬆️ TOP 5 EQUIPOS SOBREVALORADOS (Mayor sobreprecio respecto al hardware) ⬆️")
print(sobreprecios[cols].head().round(2).to_string(index=False))

🎯 DETECTOR FINANCIERO: TOP OPORTUNIDADES Y SOBREPRECIOS DE MERCADO

⬇️ TOP 5 GANGAS REALES (Mayor descuento frente al valor del hardware) ⬇️
                                                                                                                                                                                     Nombre  Precio_MXN  Precio_Justo_Modelo  Diferencia_MXN
                                                                                                                                 Laptop Gamer Lenovo Loq 16 Ryzen 7 7840hs 32gb 1tb Rtx4060   17,999.00            54,759.50      -36,760.50
                                                                                                                                  Hp Workstation Zbook 17 Core I7 32gb Ram 800gb Ssd Nvidia    6,499.00            24,443.02      -17,944.02
Portatil Gamer Lenovo Legion Go S Blanco 8 Gaming Handheld, Amd Ryzen 7 Z2 Go, 32gb De Ram, 1tb Ssd, Amd Radeon Rdna, 120hz Wuxga 1920px X 1200px, W

In [6]:
# 1. Aplicar la predicción a TODO el catálogo filtrado (X)
df_filtrado['Precio_Justo_Modelo'] = rf_opt.predict(X)
df_filtrado['Diferencia_MXN'] = df_filtrado['Precio_MXN'] - df_filtrado['Precio_Justo_Modelo']

# 2. Guardar la base de datos lista para la web
df_filtrado.to_csv('catalogo_enriquecido.csv', index=False)
print("✅ Catálogo enriquecido guardado exitosamente.")

# 3. (Opcional pero recomendado) Guardar el modelo por si la API necesita predecir algo nuevo
joblib.dump(rf_opt, 'modelo_rf_laptops.pkl')
joblib.dump(feature_cols, 'columnas_modelo.pkl')
print("✅ Modelo exportado exitosamente.")

✅ Catálogo enriquecido guardado exitosamente.
✅ Modelo exportado exitosamente.
